# Vehicle Detection, Tracking & Counting — Google Colab

This notebook is a Colab version of the uploaded `playGround.py`.

**Workflow:**
1. Install the required packages.
2. Upload your video when prompted.
3. YOLOv8 detects cars and trucks.
4. Deep SORT tracks each vehicle.
5. A horizontal counting line is drawn at the middle of the frame.
6. Vehicles crossing into the lower half are counted once per track ID.
7. The processed video is saved and displayed at the end.
8. The final video is also downloaded automatically.

The original detection classes `[2, 7]` are retained, corresponding to **car** and **truck** in the COCO YOLO model.

In [ ]:
# Install dependencies
!pip -q install ultralytics deep-sort-realtime imutils gradio

# Check that Colab can use the available accelerator
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## Gradio Interface

Run the next cell to launch the web interface.

1. Click **Upload Video**.
2. Select a video from your computer.
3. Click **Process Video**.
4. Watch the processing progress.
5. View the processed video and final vehicle count in the interface.

The notebook uses `share=True`, so Colab provides a temporary Gradio link.

In [ ]:
import cv2 as cv
from pathlib import Path
import subprocess
import torch
import gradio as gr
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLO once
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")
print(f"YOLO device: {'GPU' if device == 0 else 'CPU'}")


def draw_box(data, image, name):
    x1, y1, x2, y2, conf, class_id = data
    p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
    cv.rectangle(image, p1, p2, (0, 0, 255), 3)
    cv.putText(
        image, f"{name} {conf:.2f}", p1,
        cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2
    )


def get_details(result):
    classes = result.boxes.cls.cpu().numpy()
    conf = result.boxes.conf.cpu().numpy()
    xywh = result.boxes.xywh.cpu().numpy()
    return [(item, float(conf[i]), int(classes[i]))
            for i, item in enumerate(xywh)]


def draw_line(image):
    y = image.shape[0] // 2
    cv.line(
        image, (400, y), (image.shape[1] - 200, y),
        (0, 255, 0), thickness=6
    )


def process_video(input_path, progress=gr.Progress()):
    if input_path is None:
        raise gr.Error("Please upload a video first.")

    input_path = Path(input_path)
    raw_output = Path("/content/processed_raw.mp4")
    final_output = Path("/content/vehicle_counted_output.mp4")

    vs = cv.VideoCapture(str(input_path))
    if not vs.isOpened():
        raise gr.Error("Could not open the uploaded video.")

    fps = vs.get(cv.CAP_PROP_FPS) or 30.0
    width = int(vs.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(vs.get(cv.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(vs.get(cv.CAP_PROP_FRAME_COUNT))

    writer = cv.VideoWriter(
        str(raw_output),
        cv.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    if not writer.isOpened():
        vs.release()
        raise gr.Error("Could not create the output video.")

    # Fresh tracker for each uploaded video
    tracker = DeepSort(
        max_iou_distance=0.7,
        max_age=5,
        n_init=3,
        nms_max_overlap=1.0,
        max_cosine_distance=0.2,
        nn_budget=None,
        gating_only_position=False,
        override_track_class=None,
        embedder="mobilenet",
        half=torch.cuda.is_available(),
        bgr=True,
        embedder_gpu=torch.cuda.is_available(),
        polygon=False
    )

    counted_ids = set()
    count = 0
    frame_number = 0

    try:
        while True:
            grabbed, frame = vs.read()
            if not grabbed:
                break

            frame_number += 1

            results = model.predict(
                frame,
                stream=False,
                classes=[2, 7],  # car and truck
                verbose=False,
                device=device
            )

            draw_line(frame)
            tracks = []

            for result in results:
                if result.boxes is not None:
                    for data in result.boxes.data.cpu().tolist():
                        class_id = int(data[5])
                        draw_box(frame, data, result.names[class_id])

                    tracks = tracker.update_tracks(
                        get_details(result), frame=frame
                    )

            for track in tracks:
                if not track.is_confirmed():
                    continue

                track_id = track.track_id
                bbox = track.to_ltrb()
                x1, y1, x2, y2 = map(int, bbox)

                cv.putText(
                    frame, f"ID: {track_id}", (x1, max(30, y1)),
                    cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
                )

                # Original counting rule
                if y1 > height // 2 and track_id not in counted_ids:
                    counted_ids.add(track_id)
                    count += 1

            cv.putText(
                frame, f"Vehicle Count: {count}", (40, 70),
                cv.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4
            )

            writer.write(frame)

            if total_frames:
                progress(
                    frame_number / total_frames,
                    desc=f"Processing... Vehicles counted: {count}"
                )
    finally:
        vs.release()
        writer.release()

    # Browser-friendly MP4
    cmd = [
        "ffmpeg", "-y", "-i", str(raw_output),
        "-c:v", "libx264", "-preset", "fast", "-crf", "23",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(final_output)
    ]
    result = subprocess.run(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )

    if result.returncode != 0:
        final_output = raw_output

    return str(final_output), f"{count} vehicles detected"


with gr.Blocks(title="Vehicle Detection & Counting") as demo:
    gr.Markdown("""
    # 🚗 Vehicle Detection, Tracking & Counting

    Upload a road/highway video and click **Process Video**.

    **YOLOv8** detects cars and trucks, while **Deep SORT** tracks
    vehicles and counts each unique track crossing into the lower half.
    """)

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(
                label="Upload Video",
                sources=["upload"],
                type="filepath"
            )
            process_btn = gr.Button(
                "▶️ Process Video",
                variant="primary"
            )

        with gr.Column():
            video_output = gr.Video(
                label="Processed Video",
                interactive=False
            )
            count_output = gr.Textbox(
                label="Final Vehicle Count",
                interactive=False
            )

    process_btn.click(
        fn=process_video,
        inputs=video_input,
        outputs=[video_output, count_output]
    )

demo.queue().launch(share=True, debug=True)

## Notes

- The Colab version now uses a **Gradio interface** instead of `files.upload()`.
- The original YOLO classes `[2, 7]` are preserved for car and truck detection.
- Deep SORT assigns tracking IDs.
- The original counting rule is preserved: a unique track is counted when it enters the lower half of the frame.
- A fresh tracker is created for every uploaded video.
- The output is converted to browser-friendly H.264 MP4.
- `share=True` creates a temporary public Gradio URL.
- Long videos may take time because YOLO and Deep SORT process the video frame by frame.